### Entity & Relationship Extraction

**Using an LLM to turn paragraphs of prose into tiny structured facts (triples), so we can build a knowledge graph that answers bridge questions.**

**Example of triple**

Subject  ->  Relation  ->  Object

Malaria  ->  is a  ->  Disease

In [1]:
import os
import logging
from pathlib import Path
from dotenv import load_dotenv

os.environ['ANONYMIZED_TELEMETRY'] = 'False'                 # Silence Chroma telemetry
logging.getLogger('httpx').setLevel(logging.WARNING)          # Silence OpenAI HTTP logs

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from typing import List, Literal
import json
from tqdm import tqdm

Load the llm model

In [2]:
load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

Define the Pydantic models

In [3]:
# Allowed relations — the LLM can ONLY use these
AllowedRelation = Literal[
    'IS_A',
    'CAUSED_BY',
    'TRANSMITTED_BY',
    'AFFECTS',
    'HAS_SYMPTOM',
    'CONTROL',
    'HAS_PART',
    'LOCATED_IN',
]

# One triple: subject → relation → object
class Triple(BaseModel):
    subject: str = Field(description='The entity the fact is about, e.g., Malaria')
    relation: AllowedRelation = Field(description='The connection, from the allowed list')
    object: str = Field(description='The target entity, e.g., Disease')
    
# A container: list of triples
class TripleList(BaseModel):
    triples: List[Triple] = Field(description='All triples extracted from the passage')
    
print('Pydantic models defined')
print(f'    Allowed relations: {AllowedRelation.__args__}')

Pydantic models defined
    Allowed relations: ('IS_A', 'CAUSED_BY', 'TRANSMITTED_BY', 'AFFECTS', 'HAS_SYMPTOM', 'CONTROL', 'HAS_PART', 'LOCATED_IN')


Attach the Pydantic schema to the LLM

In [4]:
# Force the LLM to return a TripleList object
structured_llm = llm.with_structured_output(TripleList, method='json_schema')

print('Structured LLM ready')
print('   Output type will be: TripleList (a Pydantic object)')

Structured LLM ready
   Output type will be: TripleList (a Pydantic object)


Build the extraction prompt and chain

In [5]:
# Prompt tells the LLM the semantic rules; the schema enforces the format

extraction_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You extract facts from a passage and return them as triples.\n'
     '\n'
     'RULES:\n'
     '1. Only extract facts that are explicitly stated in the passage. Do not invent.\n'
     '2. Use canonical names: always "Malaria", never "malaria" or "the malaria disease".\n'
     '3. Choose the relation from the allowed list provided by the schema.\n'
     '4. If nothing in the passage is a clear fact, return an empty triples list.\n'
     '\n'
     'Focus on these types of facts:\n'
     '- What something is (category, type)\n'
     '- What causes it\n'
     '- How it spreads\n'
     '- What it affects\n'
     '- What its symptoms are\n'
     '- How it is controlled or treated'),
    ('human', 'PASSAGE:\n{context}'),
])

# Chain: prompt -> structured LLM -> TripleList object
extraction_chain = extraction_prompt | structured_llm

print('Extraction chain ready')
print('   Input:  {"context": <passage text>}')
print('   Output: TripleList object with .triples list')

Extraction chain ready
   Input:  {"context": <passage text>}
   Output: TripleList object with .triples list


Test extraction on a real malaria chunk

In [6]:
sample_text = '''
Malaria remains the foremost killer disease in Nigeria. It accounts for over 25% of
infant mortality, 30% of childhood mortality, and 11% of maternal mortality.
Malaria is mostly severe among pregnant women and children less than 5 years of age.
It is caused by Plasmodium parasites and transmitted by female Anopheles mosquitoes.
'''

# Run the extraction chain
result = extraction_chain.invoke({'context': sample_text})

print(f'    Extracted {len(result.triples)} triples:\n')

for i, t in enumerate(result.triples):
    print(f'[{i}] {t.subject} | {t.relation} | {t.object}')

    Extracted 8 triples:

[0] Malaria | IS_A | killer disease
[1] Malaria | CAUSED_BY | Plasmodium parasites
[2] Malaria | TRANSMITTED_BY | female Anopheles mosquitoes
[3] Malaria | AFFECTS | pregnant women
[4] Malaria | AFFECTS | children less than 5 years of age
[5] Malaria | AFFECTS | infant mortality
[6] Malaria | AFFECTS | childhood mortality
[7] Malaria | AFFECTS | maternal mortality


Load all chunks from the vector store

In [7]:
# Load the domain-tagged vector store
persist_dir = r'C:\Users\USER\rag_course\chroma_db_domain'

embeddings = OpenAIEmbeddings()
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings
)

# Get every chunk from the store
data = vectorstore.get(include=['documents', 'metadatas'])

chunks = data['documents']
metas = data['metadatas']

all_triples = []

for i in range(len(chunks)):
    text = chunks[i]
    meta = metas[i]
    
    # Extract triples from this chunk
    result = extraction_chain.invoke({'context': text})
    
    # Add each triple to the master list
    for t in result.triples:
        all_triples.append({
            'subject': t.subject,
            'relation': t.relation,
            'object': t.object,
            'source_file': meta.get('file_name', 'unknown'),
            'domain': meta.get('domain', 'unknown'),
        }) 
        
    # Print progress every 10 chunks
    if (i + 1) % 10 == 0:
        print(f'Processed {i + 1}/{len(chunks)} chunks — {len(all_triples)} triples so far')

print(f'\nDone. Extracted {len(all_triples)} triples total.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Processed 10/80 chunks — 18 triples so far
Processed 20/80 chunks — 40 triples so far
Processed 30/80 chunks — 47 triples so far
Processed 40/80 chunks — 49 triples so far
Processed 50/80 chunks — 57 triples so far
Processed 60/80 chunks — 73 triples so far
Processed 70/80 chunks — 82 triples so far
Processed 80/80 chunks — 84 triples so far

Done. Extracted 84 triples total.


Save triples to triples.json

In [9]:
output_path = Path(r'C:\Users\USER\rag_course\10b_graph_rag\data\triples.json')
output_path.parent.mkdir(parents=True, exist_ok=True)    # Create folder if missing

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_triples, f, indent=2, ensure_ascii=False)

print(f'Saved {len(all_triples)} triples to:')
print(f'   {output_path}')

Saved 84 triples to:
   C:\Users\USER\rag_course\10b_graph_rag\data\triples.json


In [10]:
from collections import Counter

by_file = Counter(t['source_file'] for t in all_triples)

print('Triples per source file:\n')
for name, count in by_file.most_common():
    print(f'  {count:>4}  {name}')

Triples per source file:

    69  nigeria_health_diseases_and_prevention.pdf
    15  crop_disease.pdf


Sample triples from the health PDF

In [11]:
print('=== Sample triples from the health PDF ===\n')
count = 0
for t in all_triples:
    if t['source_file'] == 'nigeria_health_diseases_and_prevention.pdf' and count < 10:
        print(f"{t['subject']} | {t['relation']} | {t['object']}")
        count += 1

=== Sample triples from the health PDF ===

Malaria | IS_A | disease of public health importance
Malaria | AFFECTS | Child survival
Malaria | CAUSED_BY | Nutritional deficiencies and illnesses
Malaria | IS_A | Killer disease
Malaria | AFFECTS | Children
Malaria | AFFECTS | Infant mortality
Malaria | AFFECTS | Childhood mortality
Malaria | AFFECTS | Maternal mortality
Malaria | HAS_SYMPTOM | 300,000 children dying each year
Malaria | HAS_SYMPTOM | over 25% of infant mortality


Sample triples from the health PDF

In [12]:
print('=== Sample triples from the health PDF ===\n')
count = 0
for t in all_triples:
    if t['source_file'] == 'nigeria_health_diseases_and_prevention.pdf' and count < 10:
        print(f"{t['subject']} | {t['relation']} | {t['object']}")
        count += 1

=== Sample triples from the health PDF ===

Malaria | IS_A | disease of public health importance
Malaria | AFFECTS | Child survival
Malaria | CAUSED_BY | Nutritional deficiencies and illnesses
Malaria | IS_A | Killer disease
Malaria | AFFECTS | Children
Malaria | AFFECTS | Infant mortality
Malaria | AFFECTS | Childhood mortality
Malaria | AFFECTS | Maternal mortality
Malaria | HAS_SYMPTOM | 300,000 children dying each year
Malaria | HAS_SYMPTOM | over 25% of infant mortality


Show all unique subjects

In [13]:
subjects = Counter(t['subject'] for t in all_triples)

print(f'Total unique subjects: {len(subjects)}\n')
print('Top 15 subjects:')
for name, count in subjects.most_common(15):
    print(f'  {count:>3}  {name}')

Total unique subjects: 9

Top 15 subjects:
   66  Malaria
    5  Necrosis
    3  Downy mildew
    2  Cassava bacterial blight
    2  Leaf blight
    2  PermaNet
    2  Cassava Mosaic Virus
    1  KidACT
    1  Fungi
